# Analisis Hasil Eksperimen Final

Notebook ini digunakan untuk menganalisis hasil eksperimen final pada tiga dataset dan tiga algoritma klasifikasi menggunakan 100 variasi train-test split.

Analisis mencakup:

- Distribusi Accuracy
- Distribusi Macro F1-score
- Mean, median, standard deviation, minimum, maximum, dan range
- Perbandingan stabilitas antaralgoritma
- Win rate
- Tie
- Konsistensi peringkat model
- Rank reversal
- Selisih performa antaralgoritma

In [222]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(
    "../../results/final/final_experiment_results.csv"
)

df.head()

,dataset,model,seed,accuracy,macro_f1
0,wine,decision_tree,1,0.944444,0.945741
1,wine,knn,1,0.972222,0.971781
2,wine,random_forest,1,1.000000,1.000000
3,wine,decision_tree,2,0.944444,0.948413
4,wine,knn,2,0.972222,0.974321


In [223]:
df.shape

(900, 5)

In [224]:
accuracy_summary = (
    df.groupby(["dataset", "model"])["accuracy"]
    .agg([
        "mean",
        "median",
        "std",
        "min",
        "max"
    ])
    .reset_index()
)

accuracy_summary["range"] = (
    accuracy_summary["max"]
    - accuracy_summary["min"]
)

accuracy_summary[
    ["mean", "median", "std", "min", "max", "range"]
] *= 100

accuracy_summary.round(2)

,dataset,model,mean,median,std,min,max,range
0,breast_cancer,decision_tree,93.11,93.86,2.67,86.84,97.37,10.53
1,breast_cancer,knn,96.62,96.49,1.49,92.98,100.00,7.02
2,breast_cancer,random_forest,96.18,96.49,1.62,92.11,99.12,7.02
3,raisin,decision_tree,81.27,81.11,2.66,75.00,88.89,13.89
4,raisin,knn,85.17,85.56,2.14,80.00,90.00,10.00
5,raisin,random_forest,86.09,86.11,2.25,80.00,91.11,11.11
6,wine,decision_tree,90.17,91.67,4.74,77.78,97.22,19.44
7,wine,knn,96.25,97.22,2.51,86.11,100.00,13.89
8,wine,random_forest,98.31,98.61,1.89,94.44,100.00,5.56


In [225]:
accuracy_summary.to_csv(
    "../../results/final/accuracy_summary.csv",
    index=False
)

In [226]:
f1_summary = (
    df.groupby(["dataset", "model"])["macro_f1"]
    .agg([
        "mean",
        "median",
        "std",
        "min",
        "max"
    ])
    .reset_index()
)

f1_summary["range"] = (
    f1_summary["max"]
    - f1_summary["min"]
)

f1_summary[
    ["mean", "median", "std", "min", "max", "range"]
] *= 100

f1_summary.round(2)

,dataset,model,mean,median,std,min,max,range
0,breast_cancer,decision_tree,92.62,93.33,2.82,86.06,97.19,11.13
1,breast_cancer,knn,96.32,96.19,1.64,92.11,100.00,7.89
2,breast_cancer,random_forest,95.89,96.19,1.74,91.71,99.05,7.35
3,raisin,decision_tree,81.24,81.09,2.67,74.77,88.88,14.10
4,raisin,knn,85.14,85.52,2.14,80.00,90.00,10.00
5,raisin,random_forest,86.07,86.11,2.25,79.99,91.11,11.12
6,wine,decision_tree,90.27,91.62,4.75,77.25,97.43,20.18
7,wine,knn,96.35,97.18,2.48,86.30,100.00,13.70
8,wine,random_forest,98.36,98.72,1.84,94.41,100.00,5.59


In [227]:
f1_summary.to_csv(
    "../../results/final/macro_f1_summary.csv",
    index=False
)

In [228]:
# Ubah data ke format wide agar setiap seed
# memiliki skor ketiga model berdampingan

accuracy_wide = df.pivot(
    index=["dataset", "seed"],
    columns="model",
    values="accuracy"
).reset_index()

accuracy_wide.head()

model,dataset,seed,decision_tree,knn,random_forest
0,breast_cancer,1,0.956140,0.973684,0.956140
1,breast_cancer,2,0.956140,0.982456,0.982456
2,breast_cancer,3,0.938596,0.973684,0.973684
3,breast_cancer,4,0.877193,0.982456,0.982456
4,breast_cancer,5,0.929825,0.973684,0.964912


In [229]:
model_columns = [
    "decision_tree",
    "knn",
    "random_forest"
]

accuracy_wide["max_score"] = (
    accuracy_wide[model_columns].max(axis=1)
)

In [230]:
def get_winner(row):
    scores = row[model_columns]

    winners = scores[
        scores == scores.max()
    ].index.tolist()

    if len(winners) == 1:
        return winners[0]

    return "tie"

accuracy_wide["winner"] = (
    accuracy_wide.apply(
        get_winner,
        axis=1
    )
)

In [231]:
win_counts = (
    accuracy_wide
    .groupby(["dataset", "winner"])
    .size()
    .reset_index(name="count")
)

win_counts

,dataset,winner,count
0,breast_cancer,decision_tree,2
1,breast_cancer,knn,41
2,breast_cancer,random_forest,27
3,breast_cancer,tie,30
4,raisin,decision_tree,2
5,raisin,knn,23
6,raisin,random_forest,62
7,raisin,tie,13
8,wine,knn,5
9,wine,random_forest,58


In [232]:
win_counts.to_csv(
    "../../results/final/accuracy_win_counts.csv",
    index=False
)

## Analisis Performance Gap

Frekuensi kemenangan suatu algoritma belum menunjukkan seberapa besar
keunggulan algoritma tersebut.

Oleh karena itu, pada setiap train-test split dihitung selisih antara
skor tertinggi dan skor tertinggi kedua.

Analisis ini digunakan untuk mengetahui apakah perubahan peringkat model
terjadi dengan perbedaan performa yang besar atau hanya dengan selisih
yang kecil.

In [233]:
def calculate_gap(row):
    scores = row[model_columns].sort_values(
        ascending=False
    )

    return scores.iloc[0] - scores.iloc[1]


accuracy_wide["performance_gap"] = (
    accuracy_wide.apply(
        calculate_gap,
        axis=1
    )
)

In [234]:
accuracy_wide["performance_gap_pp"] = (
    accuracy_wide["performance_gap"] * 100
)

In [235]:
def get_top_models(row):
    scores = row[model_columns]

    max_score = scores.max()

    winners = scores[
        scores == max_score
    ].index.tolist()

    return winners


accuracy_wide["top_models"] = (
    accuracy_wide.apply(
        get_top_models,
        axis=1
    )
)

In [236]:
accuracy_wide[
    [
        "dataset",
        "seed",
        "decision_tree",
        "knn",
        "random_forest",
        "top_models"
    ]
].head(10)

model,dataset,seed,decision_tree,knn,random_forest,top_models
0,breast_cancer,1,0.956140,0.973684,0.956140,[knn]
1,breast_cancer,2,0.956140,0.982456,0.982456,"[knn, random_forest]"
2,breast_cancer,3,0.938596,0.973684,0.973684,"[knn, random_forest]"
3,breast_cancer,4,0.877193,0.982456,0.982456,"[knn, random_forest]"
4,breast_cancer,5,0.929825,0.973684,0.964912,[knn]
5,breast_cancer,6,0.964912,0.982456,0.964912,[knn]
6,breast_cancer,7,0.929825,0.982456,0.956140,[knn]
7,breast_cancer,8,0.877193,0.973684,0.973684,"[knn, random_forest]"
8,breast_cancer,9,0.938596,0.991228,0.973684,[knn]
9,breast_cancer,10,0.938596,0.973684,0.947368,[knn]


In [237]:
gap_summary = (
    accuracy_wide
    .groupby("dataset")["performance_gap_pp"]
    .agg([
        "mean",
        "median",
        "std",
        "min",
        "max"
    ])
    .reset_index()
)

gap_summary.round(2)

,dataset,mean,median,std,min,max
0,breast_cancer,1.19,0.88,1.15,0.0,6.14
1,raisin,1.63,1.67,1.21,0.0,5.56
2,wine,2.19,2.78,2.02,0.0,8.33


In [238]:
non_tie = accuracy_wide[
    accuracy_wide["winner"] != "tie"
].copy()

gap_non_tie_summary = (
    non_tie
    .groupby("dataset")["performance_gap_pp"]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max"
    ])
    .reset_index()
)

gap_non_tie_summary.round(2)

,dataset,count,mean,median,std,min,max
0,breast_cancer,70,1.70,1.75,1.00,0.88,6.14
1,raisin,87,1.88,1.67,1.10,0.56,5.56
2,wine,63,3.48,2.78,1.41,2.78,8.33


In [239]:
winner_gap_summary = (
    non_tie
    .groupby(
        ["dataset", "winner"]
    )["performance_gap_pp"]
    .agg([
        "count",
        "mean",
        "median",
        "min",
        "max"
    ])
    .reset_index()
)

winner_gap_summary.round(2)

,dataset,winner,count,mean,median,min,max
0,breast_cancer,decision_tree,2,1.32,1.32,0.88,1.75
1,breast_cancer,knn,41,1.95,1.75,0.88,6.14
2,breast_cancer,random_forest,27,1.36,0.88,0.88,3.51
3,raisin,decision_tree,2,1.11,1.11,0.56,1.67
4,raisin,knn,23,1.64,1.67,0.56,3.89
5,raisin,random_forest,62,1.99,1.94,0.56,5.56
6,wine,knn,5,2.78,2.78,2.78,2.78
7,wine,random_forest,58,3.54,2.78,2.78,8.33


In [240]:
overall_best = (
    df.groupby(
        ["dataset", "model"]
    )["accuracy"]
    .mean()
    .reset_index()
    .sort_values(
        ["dataset", "accuracy"],
        ascending=[True, False]
    )
    .groupby("dataset")
    .first()
    .reset_index()
)

overall_best

,dataset,model,accuracy
0,breast_cancer,knn,0.966228
1,raisin,random_forest,0.860889
2,wine,random_forest,0.983056


In [241]:
overall_best_map = dict(
    zip(
        overall_best["dataset"],
        overall_best["model"]
    )
)

In [242]:
def classify_split(row):

    best_model = overall_best_map[
        row["dataset"]
    ]

    top_models = row["top_models"]

    # Pemenang tunggal sama dengan pemenang overall
    if (
        len(top_models) == 1
        and top_models[0] == best_model
    ):
        return "consistent"

    # Seri dan model overall ikut dalam seri
    elif best_model in top_models:
        return "tie_with_overall_best"

    # Model overall tidak menjadi yang terbaik
    else:
        return "rank_reversal"


accuracy_wide["ranking_status"] = (
    accuracy_wide.apply(
        classify_split,
        axis=1
    )
)

In [243]:
rank_reversal_summary = (
    accuracy_wide
    .groupby(
        ["dataset", "ranking_status"]
    )
    .size()
    .reset_index(name="count")
)

rank_reversal_summary

,dataset,ranking_status,count
0,breast_cancer,consistent,41
1,breast_cancer,rank_reversal,31
2,breast_cancer,tie_with_overall_best,28
3,raisin,consistent,62
4,raisin,rank_reversal,25
5,raisin,tie_with_overall_best,13
6,wine,consistent,58
7,wine,rank_reversal,5
8,wine,tie_with_overall_best,37


In [244]:
# Ubah Macro F1 ke format wide

f1_wide = df.pivot(
    index=["dataset", "seed"],
    columns="model",
    values="macro_f1"
).reset_index()

f1_wide.head()

model,dataset,seed,decision_tree,knn,random_forest
0,breast_cancer,1,0.953106,0.971277,0.952638
1,breast_cancer,2,0.952638,0.980956,0.980956
2,breast_cancer,3,0.932981,0.971277,0.971583
3,breast_cancer,4,0.871498,0.980956,0.980956
4,breast_cancer,5,0.926570,0.971863,0.962302


In [245]:
model_columns = [
    "decision_tree",
    "knn",
    "random_forest"
]

In [246]:
def get_top_models(row):
    scores = row[model_columns]

    max_score = scores.max()

    winners = scores[
        scores == max_score
    ].index.tolist()

    return winners


f1_wide["top_models"] = (
    f1_wide.apply(
        get_top_models,
        axis=1
    )
)

In [247]:
def get_winner(row):
    winners = row["top_models"]

    if len(winners) == 1:
        return winners[0]

    return "tie"


f1_wide["winner"] = (
    f1_wide.apply(
        get_winner,
        axis=1
    )
)

In [248]:
f1_win_counts = (
    f1_wide
    .groupby(["dataset", "winner"])
    .size()
    .reset_index(name="count")
)

f1_win_counts

,dataset,winner,count
0,breast_cancer,decision_tree,2
1,breast_cancer,knn,42
2,breast_cancer,random_forest,44
3,breast_cancer,tie,12
4,raisin,decision_tree,2
5,raisin,knn,25
6,raisin,random_forest,72
7,raisin,tie,1
8,wine,decision_tree,1
9,wine,knn,7


In [249]:
def calculate_gap(row):
    scores = row[model_columns].sort_values(
        ascending=False
    )

    return scores.iloc[0] - scores.iloc[1]


f1_wide["performance_gap"] = (
    f1_wide.apply(
        calculate_gap,
        axis=1
    )
)

f1_wide["performance_gap_pp"] = (
    f1_wide["performance_gap"] * 100
)

In [250]:
f1_gap_summary = (
    f1_wide
    .groupby("dataset")["performance_gap_pp"]
    .agg([
        "mean",
        "median",
        "std",
        "min",
        "max"
    ])
    .reset_index()
)

f1_gap_summary.round(2)

,dataset,mean,median,std,min,max
0,breast_cancer,1.28,0.97,1.21,0.0,6.41
1,raisin,1.64,1.66,1.21,0.0,5.55
2,wine,2.15,2.57,1.96,0.0,8.36


In [251]:
f1_non_tie = f1_wide[
    f1_wide["winner"] != "tie"
].copy()

f1_gap_non_tie_summary = (
    f1_non_tie
    .groupby("dataset")["performance_gap_pp"]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max"
    ])
    .reset_index()
)

f1_gap_non_tie_summary.round(2)

,dataset,count,mean,median,std,min,max
0,breast_cancer,88,1.45,1.02,1.19,0.02,6.41
1,raisin,99,1.65,1.66,1.20,0.00,5.55
2,wine,71,3.03,2.82,1.65,0.03,8.36


In [252]:
f1_overall_best = (
    df.groupby(
        ["dataset", "model"]
    )["macro_f1"]
    .mean()
    .reset_index()
    .sort_values(
        ["dataset", "macro_f1"],
        ascending=[True, False]
    )
    .groupby("dataset")
    .first()
    .reset_index()
)

f1_overall_best

,dataset,model,macro_f1
0,breast_cancer,knn,0.963181
1,raisin,random_forest,0.860691
2,wine,random_forest,0.983553


In [253]:
f1_overall_best_map = dict(
    zip(
        f1_overall_best["dataset"],
        f1_overall_best["model"]
    )
)

In [254]:
def classify_f1_split(row):

    best_model = f1_overall_best_map[
        row["dataset"]
    ]

    top_models = row["top_models"]

    if (
        len(top_models) == 1
        and top_models[0] == best_model
    ):
        return "consistent"

    elif best_model in top_models:
        return "tie_with_overall_best"

    else:
        return "rank_reversal"


f1_wide["ranking_status"] = (
    f1_wide.apply(
        classify_f1_split,
        axis=1
    )
)

In [255]:
f1_rank_reversal_summary = (
    f1_wide
    .groupby(
        ["dataset", "ranking_status"]
    )
    .size()
    .reset_index(name="count")
)

f1_rank_reversal_summary

,dataset,ranking_status,count
0,breast_cancer,consistent,42
1,breast_cancer,rank_reversal,46
2,breast_cancer,tie_with_overall_best,12
3,raisin,consistent,72
4,raisin,rank_reversal,27
5,raisin,tie_with_overall_best,1
6,wine,consistent,63
7,wine,rank_reversal,8
8,wine,tie_with_overall_best,29


In [256]:
f1_win_counts.to_csv(
    "../../results/final/f1_win_counts.csv",
    index=False
)

f1_gap_summary.to_csv(
    "../../results/final/f1_gap_summary.csv",
    index=False
)

f1_gap_non_tie_summary.to_csv(
    "../../results/final/f1_gap_non_tie_summary.csv",
    index=False
)

f1_rank_reversal_summary.to_csv(
    "../../results/final/f1_rank_reversal_summary.csv",
    index=False
)

In [257]:
comparison_metrics = accuracy_wide[
    ["dataset", "seed", "top_models"]
].copy()

comparison_metrics = comparison_metrics.rename(
    columns={
        "top_models": "accuracy_top_models"
    }
)

comparison_metrics["f1_top_models"] = (
    f1_wide["top_models"]
)

comparison_metrics.head()

model,dataset,seed,accuracy_top_models,f1_top_models
0,breast_cancer,1,[knn],[knn]
1,breast_cancer,2,"[knn, random_forest]","[knn, random_forest]"
2,breast_cancer,3,"[knn, random_forest]",[random_forest]
3,breast_cancer,4,"[knn, random_forest]","[knn, random_forest]"
4,breast_cancer,5,[knn],[knn]


In [258]:
def same_top_models(row):
    return set(
        row["accuracy_top_models"]
    ) == set(
        row["f1_top_models"]
    )


comparison_metrics["same_winner_set"] = (
    comparison_metrics.apply(
        same_top_models,
        axis=1
    )
)

In [259]:
metric_agreement = (
    comparison_metrics
    .groupby("dataset")["same_winner_set"]
    .agg([
        "sum",
        "count",
        "mean"
    ])
    .reset_index()
)

metric_agreement["agreement_percent"] = (
    metric_agreement["mean"] * 100
)

metric_agreement.round(2)

,dataset,sum,count,mean,agreement_percent
0,breast_cancer,82,100,0.82,82.0
1,raisin,88,100,0.88,88.0
2,wine,92,100,0.92,92.0


In [260]:
metric_disagreement = comparison_metrics[
    comparison_metrics["same_winner_set"] == False
].copy()

metric_disagreement.groupby(
    "dataset"
).size()

dataset
breast_cancer    18
raisin           12
wine              8
dtype: int64

In [261]:
metric_disagreement.head(20)

model,dataset,seed,accuracy_top_models,f1_top_models,same_winner_set
2,breast_cancer,3,"[knn, random_forest]",[random_forest],False
7,breast_cancer,8,"[knn, random_forest]",[random_forest],False
14,breast_cancer,15,"[decision_tree, random_forest]",[random_forest],False
30,breast_cancer,31,"[knn, random_forest]",[random_forest],False
32,breast_cancer,33,"[knn, random_forest]",[random_forest],False
38,breast_cancer,39,"[knn, random_forest]",[random_forest],False
49,breast_cancer,50,"[decision_tree, random_forest]",[random_forest],False
50,breast_cancer,51,"[knn, random_forest]",[random_forest],False
51,breast_cancer,52,"[knn, random_forest]",[random_forest],False
53,breast_cancer,54,"[knn, random_forest]",[random_forest],False


In [262]:
def classify_metric_disagreement(row):

    acc = set(row["accuracy_top_models"])
    f1 = set(row["f1_top_models"])

    # Benar-benar sama
    if acc == f1:
        return "same"

    # Salah satu metrik menghasilkan tie
    # dan metrik lain memilih anggota dari tie tersebut
    if (
        (len(acc) > 1 and len(f1) == 1 and f1.issubset(acc))
        or
        (len(f1) > 1 and len(acc) == 1 and acc.issubset(f1))
    ):
        return "tie_resolution"

    # Pemenang / set pemenang benar-benar berbeda
    return "metric_reversal"


comparison_metrics["metric_relation"] = (
    comparison_metrics.apply(
        classify_metric_disagreement,
        axis=1
    )
)

In [263]:
metric_relation_summary = (
    comparison_metrics
    .groupby(
        ["dataset", "metric_relation"]
    )
    .size()
    .reset_index(name="count")
)

metric_relation_summary

,dataset,metric_relation,count
0,breast_cancer,same,82
1,breast_cancer,tie_resolution,18
2,raisin,same,88
3,raisin,tie_resolution,12
4,wine,same,92
5,wine,tie_resolution,8


In [264]:
metric_relation_summary["percent"] = (
    metric_relation_summary["count"]
)

metric_relation_summary

,dataset,metric_relation,count,percent
0,breast_cancer,same,82,82
1,breast_cancer,tie_resolution,18,18
2,raisin,same,88,88
3,raisin,tie_resolution,12,12
4,wine,same,92,92
5,wine,tie_resolution,8,8


In [265]:
true_metric_reversal = comparison_metrics[
    comparison_metrics["metric_relation"]
    == "metric_reversal"
]

true_metric_reversal    

model,dataset,seed,accuracy_top_models,f1_top_models,same_winner_set,metric_relation


In [266]:
metric_relation_summary.to_csv(
    "../../results/final/metric_relation_summary.csv",
    index=False
)

rank_reversal_summary.to_csv(
    "../../results/final/accuracy_rank_reversal_summary.csv",
    index=False
)

gap_summary.to_csv(
    "../../results/final/accuracy_gap_summary.csv",
    index=False
)

gap_non_tie_summary.to_csv(
    "../../results/final/accuracy_gap_non_tie_summary.csv",
    index=False
)

winner_gap_summary.to_csv(
    "../../results/final/accuracy_winner_gap_summary.csv",
    index=False
)